In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb

df = pd.read_csv('../data/ald_process_data.csv')
cat_cols = ['material', 'precursor', 'oxidant']
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[f'{col}_enc'] = le.fit_transform(df[col])
    encoders[col] = le

feature_cols = ['material_enc', 'precursor_enc', 'oxidant_enc',
                'temperature', 'pulse_time', 'purge_time']

model = xgb.XGBRegressor()
model.load_model('../data/xgb_process_model.json')

importance = pd.Series(model.feature_importances_, index=feature_cols)
print("=== GPC 모델 Feature Importance ===")
print(importance.sort_values(ascending=False).round(4))

## ALD 물리 해석: 공정 파라미터가 GPC를 결정하는 방식

### 자기 제한적 반응 (Self-Limiting Reaction)
ALD의 핵심: 전구체가 표면에 포화 흡착 후 더 이상 반응하지 않음
→ pulse_time이 포화 이상이면 GPC 일정 (CVD와 근본적 차이)

### ALD 온도 윈도우 (Temperature Window)
| 구간 | 현상 | GPC 거동 |
|------|------|----------|
| T < T_min | 전구체 흡착 불완전 (활성화 에너지 부족) | GPC 감소 |
| T_min ~ T_max | **ALD 윈도우**: 포화 흡착 안정 | GPC 평탄 |
| T > T_max | 전구체 열분해 (CVD 모드 진입) | GPC 증가 |

HfO₂ ALD 윈도우: 150~300°C (TDMAH + H₂O 기준)
Al₂O₃ ALD 윈도우: 100~280°C (TMA + H₂O 기준)
ZrO₂ ALD 윈도우: 150~280°C (ZrCl₄ + H₂O 기준)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
materials = ['HfO2', 'Al2O3', 'ZrO2']
colors    = ['steelblue', 'darkorange', 'seagreen']
windows   = {'HfO2': (150, 300), 'Al2O3': (100, 280), 'ZrO2': (150, 280)}

for ax, mat, col in zip(axes, materials, colors):
    sub = df[df['material'] == mat].sort_values('temperature')
    ax.scatter(sub['temperature'], sub['gpc'], alpha=0.5, s=20, color=col)

    lo, hi = windows[mat]
    ax.axvspan(lo, hi, alpha=0.1, color='green', label='ALD window')
    ax.axvline(lo, color='green', linestyle='--', linewidth=1, alpha=0.7)
    ax.axvline(hi, color='green', linestyle='--', linewidth=1, alpha=0.7)

    ax.set_xlabel('Temperature (°C)')
    ax.set_ylabel('GPC (Å/cycle)')
    ax.set_title(f'{mat} — Temperature Window')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../data/ald_temperature_window.png', dpi=150)
plt.show()
print("ALD 온도 윈도우 시각화 저장")

In [ ]:
model_eg = xgb.XGBRegressor()
model_eg.load_model('../data/xgb_bandgap_model.json')
model_k  = xgb.XGBRegressor()
model_k.load_model('../data/xgb_dielectric_model.json')

imp_eg  = pd.Series(model_eg.feature_importances_, index=model_eg.get_booster().feature_names)
imp_k   = pd.Series(model_k.feature_importances_,  index=model_k.get_booster().feature_names)
imp_gpc = importance

print("=" * 60)
print("Phase 1/2/3 Feature Importance 1위 비교")
print("=" * 60)
print(f"Phase 1 (밴드갭):  {imp_eg.idxmax()}  ({imp_eg.max():.4f})")
print(f"Phase 2 (유전율):  {imp_k.idxmax()}   ({imp_k.max():.4f})")
print(f"Phase 3 (GPC):     {imp_gpc.idxmax()} ({imp_gpc.max():.4f})")
print()
print("→ 세 모델이 서로 다른 feature와 물리를 포착함")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
data = [
    (imp_eg,  'Phase 1: 밴드갭\n(소재 스크리닝)', 'steelblue'),
    (imp_k,   'Phase 2: 유전율\n(Pareto 스크리닝)', 'darkorange'),
    (imp_gpc, 'Phase 3: GPC\n(공정 최적화)', 'seagreen'),
]
for ax, (imp, title, col) in zip(axes, data):
    imp.sort_values(ascending=False).head(8)[::-1].plot(kind='barh', ax=ax, color=col)
    ax.set_title(title)
    ax.set_xlabel('Importance (gain)')

plt.tight_layout()
plt.savefig('../data/ald_phase123_comparison.png', dpi=150)
plt.show()

## 면접 최종 스크립트 — 3단계 통합 (3분)

Phase 1에서는 153,000개 소재의 밴드갭을 XGBoost로 예측했습니다.
1위 feature는 NdValence — d 오비탈 점유 상태. 결정장 이론으로 설명합니다.

Phase 2에서는 유전율(k)도 같은 파이프라인으로 예측했습니다.
1위 feature는 NdUnfilled — 빈 d 오비탈. Clausius-Mossotti 이론으로 설명합니다.
NdValence와 NdUnfilled — 같은 d 오비탈이지만 반대 신호.
Pareto front로 두 조건을 동시에 만족하는 산화물 후보 381개를 선정했습니다.

Phase 3에서는 선정 소재의 ALD 공정 파라미터로 GPC를 예측했습니다.
1위 feature는 temperature / material — ALD 온도 윈도우가 GPC를 지배합니다.
문헌 데이터 기반으로, 공정 조건 입력 시 GPC를 사전 예측해
실험 횟수를 줄이고 공정 윈도우를 미리 좁힐 수 있습니다.

세 단계 모두 XGBoost를 썼지만, 각각 완전히 다른 물리를 포착했습니다:
소재 → 공정 → 수율까지, 반도체 개발 사이클 전반에 데이터 기반 접근이 가능합니다.